# XGBoost

The goal of this notebook is to see whether we can use XGBoost to predict future power production.

CURRENTLY WORKING ON THIS --- nothing here is to be trusted! I'm learning about xgboost for time series and a lot of the intermediate code will not be mine.

- should likely include the shifted series and/or "lags" (mimics ARIMA, in a way)
- can include other temporal variables, like time of year
- can include exogenous variables

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sn
import matplotlib.pyplot as plt
from extract_and_clean import Clean
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV

individual_data = pd.read_csv('../../../../data_ds_project/systems/prize/new_inverter_names_for_prize_cleaned_power.csv')
meters_df = pd.DataFrame({
    'system_id' : [2105, 9068], 
    'old_name' : ['meter','meter'], 
    'new_name' : ['000','000']
})
all_datas = pd.concat([individual_data, meters_df], ignore_index = True)




### Some basic functions to help turn time series data into data we can use for XGBoost
Include:
- shifting columns
- finding lags

In [52]:
# take time series data and add shifted columns
def shift_columns(data, n_shift=[1]):
    """Shift all columns forward by all values in n_shift. Output has same length as data.

    Args:
        data (DataFrame, series, or list): time series data
        n_shift (list or int): all the quantities shifted by. Initialized to [1]

    Returns:
        DataFrame: the shifted columns ONLY. Does not include original DataFrame
    """
    df_og = pd.DataFrame(data)
    df = pd.DataFrame() #will return this DataFrame
    cols=[] #the columns to append
    #make sure n_shift is a list
    if isinstance(n_shift, int):
        n_shift = [n_shift]
    
    for i in n_shift:
        shifted = df_og.shift(i)

        shifted = shifted.rename(
            columns=lambda col: f"{col}_shifted_by_{i}"
        )

        cols.append(shifted)

    df = pd.concat(cols, axis = 1)
    return df

In [ ]:
# calculate lags: difference between this entry and previous entry
def calc_lags(data, n_lags = [1]):
    """outputs lags of timeseries data

    Args:
        data (DataFrame): Data to calculate lags of
        n_lags (list, optional): List of all lags to be calculated. Defaults to [1].

    Returns:
        DataFrame: lags ONLY
    """

    df_og = pd.DataFrame(data)
    if isinstance(n_lags, int):
        n_lags = [n_lags]
    
    new_columns = []
    for col in df_og.columns:
        column = df_og[col]
        print(f"column = \n{column}")
        #now make a bunch of columns representing the lags
        for i in n_lags:
            new_col = column - column.shift(i)
            print(f"new_col = \n{new_col}")
            new_col.name = f"{col}_lag_{i}"
            new_columns.append(new_col)

    return pd.concat(new_columns, axis = 1)



In [ ]:
def make_features(data: pd.DataFrame, target='power', lags=[1], n_shift=[1]) -> pd.DataFrame:
    """_summary_

    Args:
        data (pd.DataFrame): _description_
        target (str, optional): _description_. Defaults to 'power'.
        lags (list, optional): _description_. Defaults to [1].
        n_shift (list, optional): _description_. Defaults to [1].

    Returns:
        pd.DataFrame: dataframe containing all features
    """
    df=data.copy()

    #make sure 'time' column is in datetime format
    df['time'] = pd.datetime(df['time'])
 
    # incorporate lags
    for lag in lags:
        df[f"lag_{lag}"] = df[target].shift(lag)

    # incorporate shifts
    for shift in n_shift:
        df[f"shift_{shift}"] = df[target].shift(shift)

    # incorporate month of the year -- promotes seasonality
    df['month'] = df['time'].dt.month

    return df

    


Let's try running GXBoost naively on a single file of ONLY time-series data

In [ ]:
#enter file


#insert missing days
#insert proportion column, remove power column
#do calculations -- add columns and all
#drop days where info is missing  .dropnan()
data = pd.DataFrame() #we will define this later -- make this be the WHOLE dataframe


#train test split
train_size = int(len(data)*0.8)

X = data.drop(columns = ['proportion'])
y = data['proportion']

X_train = X.iloc[:train_size]
y_train = y.iloc[:train_size]

X_test = X.iloc[train_size:]
y_test = y.iloc[train_size:]


#now do CV for hyperparameter tuning
#hyperparameters we need to consider: n_estimators, max_depth, learning_rate, early_stopping_rounds, subsample
#will do RandomizedSearchCV
param_dist = {
    'n_estimators': np.random.randint(200, 800),
    'max_depth': np.random.randint(3, 8),
    'learning_rate': np.random.uniform(0.01, 0.1),
    'subsample': np.random.uniform(0.6, 0.4),
    'colsample_bytree': np.random.uniform(0.6, 0.4),
    'reg_lambda': np.random.uniform(1, 20)
}

randomized_search = RandomizedSearchCV(
    XGBRegressor(),
    param_distributions=param_dist,
    n_iter=100,
    scoring = 'neg_mean_squared_error'
)

randomized_search.fit(X_train, y_train)

best_parameters = randomized_search.best_params_
best_score = randomized_search.best_score_


